# 충남대학교 학내 정보 Q&A 시스템

**자연어처리 텀프로젝트 — 202202497**

RAG + QLoRA 파인튜닝 기반 학내 정보 질의응답 시스템

- Base 모델: `Qwen/Qwen2.5-7B-Instruct` (4bit NF4)
- 임베딩: `BAAI/bge-m3`
- 벡터 DB: ChromaDB
- 파인튜닝: QLoRA (r=16, alpha=32)

> 이 노트북은 "모두 실행"만으로 결과를 확인할 수 있습니다.

## 1. 환경 설정

In [ ]:
%%time
# 의존성 설치 (5분 이내)
!pip install -q transformers accelerate bitsandbytes peft \
    sentence-transformers chromadb gradio torch datasets \
    fastapi uvicorn

In [ ]:
import os
import json
import time
import torch
import random

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. 데이터 다운로드

벡터 DB 인덱스와 LoRA 어댑터를 HuggingFace Hub에서 다운로드합니다.

> **TODO**: 학습 완료 후 HF Hub repo ID를 채워 넣으세요.

In [ ]:
# HF Hub repo ID
HF_REPO = "adoveflash/cnu-qa-system"

from huggingface_hub import snapshot_download

# LoRA 어댑터 다운로드
ADAPTER_PATH = "models/lora_adapter"
if not os.path.exists(ADAPTER_PATH):
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=".",
        allow_patterns=["models/lora_adapter/*"],
    )
    print(f"LoRA 어댑터 다운로드 완료: {ADAPTER_PATH}")
else:
    print(f"LoRA 어댑터 이미 존재: {ADAPTER_PATH}")

# 벡터 DB 인덱스 다운로드
VECTOR_DB_PATH = "data/vector_db"
if not os.path.exists(VECTOR_DB_PATH):
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=".",
        allow_patterns=["data/vector_db/*"],
    )
    print(f"벡터 DB 다운로드 완료: {VECTOR_DB_PATH}")
else:
    print(f"벡터 DB 이미 존재: {VECTOR_DB_PATH}")

# eval 데이터 다운로드
EVAL_PATH = "data/qa/eval.jsonl"
if not os.path.exists(EVAL_PATH):
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=".",
        allow_patterns=["data/qa/eval.jsonl"],
    )
    print(f"eval 데이터 다운로드 완료: {EVAL_PATH}")
else:
    print(f"eval 데이터 이미 존재: {EVAL_PATH}")

## 3. 모델 로드

Base 모델(4bit NF4) + LoRA 어댑터를 로드합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# 4bit NF4 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 토크나이저
print("[1/3] 토크나이저 로드")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 베이스 모델
print("[2/3] 베이스 모델 로드 (4bit)")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# LoRA 어댑터 적용
print("[3/3] LoRA 어댑터 적용")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

vram_gb = torch.cuda.memory_reserved() / 1024**3
print(f"\n모델 로드 완료 — VRAM 사용: {vram_gb:.2f} GB")

## 4. RAG 검색기 초기화

bge-m3 임베딩 모델과 ChromaDB 벡터 인덱스를 로드합니다.

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

# 임베딩 모델
print("임베딩 모델 로드: BAAI/bge-m3")
embed_model = SentenceTransformer("BAAI/bge-m3")

# ChromaDB
db_path = Path(VECTOR_DB_PATH)
client = chromadb.PersistentClient(path=str(db_path))
collection = client.get_collection("cnu_chunks")
print(f"벡터 DB 로드 완료: {collection.count()}개 청크")


def retrieve(query, top_k=5):
    """질문과 유사한 청크를 검색한다."""
    query_emb = embed_model.encode([query]).tolist()[0]
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    return results


def build_context(query, top_k=5):
    """질문에 대한 컨텍스트 문자열과 출처 URL을 생성한다."""
    results = retrieve(query, top_k)
    context_parts = []
    urls = []
    for i in range(len(results["ids"][0])):
        title = results["metadatas"][0][i]["title"]
        text = results["documents"][0][i]
        url = results["metadatas"][0][i]["url"]
        context_parts.append(f"[참고{i+1}] {title}\n{text}")
        if url not in urls:
            urls.append(url)
    return "\n\n".join(context_parts), urls


# 검색 테스트
test_results = retrieve("졸업 요건이 어떻게 되나요?", top_k=3)
for i, doc in enumerate(test_results["documents"][0]):
    print(f"  [{i+1}] {doc[:80]}...")

## 5. 추론 함수 정의

In [ ]:
SYSTEM_PROMPT = (
    "당신은 충남대학교 학내 정보 안내 도우미입니다. "
    "주어진 참고 자료를 바탕으로 정확하게 답변하세요. "
    "참고 자료에 없는 내용은 '확인되지 않은 정보입니다'라고 답하세요. "
    "답변 끝에 출처 URL을 포함하세요."
)


def generate_answer(question, max_new_tokens=256):
    """RAG 검색 → LLM 답변 생성 파이프라인."""
    context, urls = build_context(question)

    if not context:
        return "관련 정보를 찾을 수 없습니다."

    user_msg = f"참고 자료:\n{context}\n\n질문: {question}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    torch.manual_seed(SEED)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # 출처 URL 추가
    if urls and not any(url in answer for url in urls):
        answer += "\n\n출처:\n" + "\n".join(f"- {url}" for url in urls)

    return answer


# 추론 테스트
test_answer = generate_answer("수강신청은 어떻게 하나요?")
print(test_answer)

## 6. Gradio UI 실행

In [ ]:
import gradio as gr

with gr.Blocks(title="충남대학교 학내 정보 Q&A") as demo:
    gr.Markdown(
        """
        # 충남대학교 학내 정보 Q&A 시스템
        학사, 장학금, 취업, 컴퓨터융합학부 정보를 질문해보세요.
        """
    )
    with gr.Row():
        question_input = gr.Textbox(
            label="질문",
            placeholder="예: 졸업 요건이 어떻게 되나요?",
            lines=2,
        )
    submit_btn = gr.Button("질문하기", variant="primary")
    answer_output = gr.Textbox(label="답변", lines=10, interactive=False)

    gr.Examples(
        examples=[
            "컴퓨터융합학부 졸업 요건이 어떻게 되나요?",
            "수강신청은 언제 하나요?",
            "인재개발원에서 하는 취업 프로그램이 뭐가 있나요?",
            "장학금 신청은 어떻게 하나요?",
        ],
        inputs=question_input,
    )

    submit_btn.click(fn=generate_answer, inputs=question_input, outputs=answer_output)
    question_input.submit(fn=generate_answer, inputs=question_input, outputs=answer_output)

demo.launch(share=True)

## 6-1. REST API 엔드포인트

Gradio 웹앱과 함께 REST API도 제공합니다.

- `POST /api/ask` — 질문을 보내면 답변을 반환합니다.
- `GET /health` — 서버 상태를 확인합니다.

API 테스트 예시:
```python
import requests
resp = requests.post("http://localhost:7860/api/ask", json={"question": "졸업 요건이 어떻게 되나요?"})
print(resp.json())
```

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

# FastAPI 앱 생성
api_app = FastAPI(
    title="충남대학교 학내 정보 Q&A API",
    description="충남대 학사·장학금·취업·컴공과 정보를 RAG 기반으로 답변하는 API",
    version="1.0.0",
)


class QuestionRequest(BaseModel):
    question: str


class AnswerResponse(BaseModel):
    question: str
    answer: str
    sources: list[str]


@api_app.get("/health")
def health_check():
    return {"status": "ok"}


@api_app.post("/api/ask", response_model=AnswerResponse)
def ask_question(req: QuestionRequest):
    question = req.question.strip()
    if not question:
        return AnswerResponse(question=question, answer="질문을 입력해주세요.", sources=[])

    context, urls = build_context(question)
    if not context:
        return AnswerResponse(question=question, answer="관련 정보를 찾을 수 없습니다.", sources=[])

    answer = generate_answer(question)
    return AnswerResponse(question=question, answer=answer, sources=urls)


# Gradio + FastAPI 통합 실행 (기존 demo 종료 후 재실행)
demo.close()
demo.launch(share=True, app=api_app)
print("\nAPI 문서: <공유 URL>/docs")

## 7. 평가: 100개 일괄 추론 + LLM Judge

In [ ]:
# eval.jsonl 로드
with open(EVAL_PATH, encoding="utf-8") as f:
    eval_data = [json.loads(line) for line in f if line.strip()]

print(f"평가 샘플 수: {len(eval_data)}개")

# 일괄 추론
results = []
for i, qa in enumerate(eval_data, 1):
    print(f"[{i}/{len(eval_data)}] {qa['question'][:50]}...", end=" ", flush=True)
    start = time.time()
    prediction = generate_answer(qa["question"])
    elapsed_ms = (time.time() - start) * 1000
    results.append({
        "question": qa["question"],
        "reference": qa["answer"],
        "prediction": prediction,
        "latency_ms": round(elapsed_ms, 1),
    })
    print(f"{elapsed_ms:.0f}ms")

# latency 통계
latencies = [r["latency_ms"] for r in results]
print(f"\n평균 지연: {sum(latencies)/len(latencies):.0f}ms")
print(f"최소/최대: {min(latencies):.0f}ms / {max(latencies):.0f}ms")

In [ ]:
import re

def llm_judge(question, reference, prediction):
    """LLM으로 답변 품질을 1~5점으로 평가한다."""
    judge_prompt = f"""다음 질문에 대한 모델 답변을 평가하세요.

질문: {question}
정답: {reference}
모델 답변: {prediction}

평가 기준:
- 5점: 정답과 동일하거나 더 상세한 정확한 답변
- 4점: 핵심 내용은 맞지만 일부 세부사항 누락
- 3점: 부분적으로 맞지만 중요한 내용 누락 또는 부정확
- 2점: 대부분 부정확하거나 관련 없는 답변
- 1점: 완전히 틀리거나 답변 거부

반드시 아래 JSON 형식으로만 응답하세요:
{{"score": 점수, "reason": "이유"}}"""

    messages = [{"role": "user", "content": judge_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    torch.manual_seed(SEED)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=128, do_sample=False,
        )
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    resp = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    match = re.search(r"\{.*\}", resp, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            return int(parsed.get("score", 0)), str(parsed.get("reason", ""))
        except (json.JSONDecodeError, ValueError):
            pass
    return 0, "파싱 실패"


# LLM Judge 평가
print("LLM Judge 평가 시작\n")
for i, r in enumerate(results, 1):
    print(f"[{i}/{len(results)}]", end=" ", flush=True)
    score, reason = llm_judge(r["question"], r["reference"], r["prediction"])
    r["judge_score"] = score
    r["judge_reason"] = reason
    print(f"점수: {score}/5")

# 결과 요약
scores = [r["judge_score"] for r in results if r["judge_score"] > 0]
avg_score = sum(scores) / len(scores) if scores else 0
avg_latency = sum(latencies) / len(latencies)

print(f"\n{'='*50}")
print(f"평가 결과 요약")
print(f"{'='*50}")
print(f"평균 점수: {avg_score:.2f}/5.0 ({len(scores)}/{len(results)}건 평가)")
print(f"평균 지연: {avg_latency:.0f}ms")
print(f"점수 분포: " + ", ".join(f"{s}점:{sum(1 for r in results if r['judge_score']==s)}건" for s in range(5, 0, -1)))

In [ ]:
# 결과 저장
output_path = "data/eval_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump({
        "avg_score": round(avg_score, 2),
        "avg_latency_ms": round(avg_latency, 1),
        "total": len(results),
        "scored": len(scores),
        "results": results,
    }, f, ensure_ascii=False, indent=2)
print(f"상세 결과 저장: {output_path}")

# 샘플 출력
print(f"\n{'='*50}")
print("샘플 결과 (상위 5개)")
print(f"{'='*50}")
for r in results[:5]:
    print(f"\nQ: {r['question']}")
    print(f"정답: {r['reference'][:100]}...")
    print(f"예측: {r['prediction'][:100]}...")
    print(f"점수: {r['judge_score']}/5 | 지연: {r['latency_ms']}ms")